# Notebook 04 — Audit and Clean News Data

## Goal
Inspect raw GDELT noise. Apply cleaning rules step by step, showing rows removed at each step.
Save an approved cleaned news file.

**Every cleaning step is visible here.** Do not move logic to helper scripts.

---

## What can go wrong
- Language filter may remove relevant non-English titles about US markets
- Deduplication by URL may be too aggressive (legitimate repeat coverage)
- Min title length may remove genuine short headlines
- COVID months may have unusual coverage patterns

---

## What you must inspect
- Random title samples before and after cleaning
- Rows removed at each step — is any step removing too many?
- Final cleaned file: are titles readable and relevant?

In [ ]:
import pathlib, json, datetime
import pandas as pd
import numpy as np
import yaml

DRIVE_ROOT = pathlib.Path('/content/drive/MyDrive/labor_news_rag_project')
REPO_DIR = pathlib.Path('/content/economic-news-labor-rag')

RAW_NEWS_PATH = DRIVE_ROOT / 'data' / 'raw' / 'gdelt' / 'news_raw.parquet'

approval_02 = DRIVE_ROOT / 'approvals' / '02_raw_gdelt_approved.json'
if not approval_02.exists():
    raise FileNotFoundError('STOP: Notebook 02 has not been approved. Run NB 02 first.')
with open(approval_02) as f:
    ap = json.load(f)
assert ap['approved'], 'Notebook 02 not approved.'

with open(REPO_DIR / 'configs' / 'cleaning_rules_news.yaml') as f:
    rules = yaml.safe_load(f)

news_raw = pd.read_parquet(RAW_NEWS_PATH)
print(f'Raw news loaded: {news_raw.shape}')
print(f'Columns: {list(news_raw.columns)}')

In [ ]:
def show_random_titles(df, query_group=None, n=20, label=''):
    sub = df if query_group is None else df[df['query_group'] == query_group]
    sub = sub.dropna(subset=['title'])
    sample = sub.sample(min(n, len(sub)), random_state=42)
    tag = f'[{query_group}]' if query_group else '[all]'
    print(f'\n=== Random titles {tag} {label} (n={len(sample)}) ===')
    for i, row in enumerate(sample.itertuples(), 1):
        print(f'  {i:2d}. {row.title}')

def print_step(df_before, df_after, step_name):
    removed = len(df_before) - len(df_after)
    pct = removed / len(df_before) * 100 if len(df_before) > 0 else 0
    print(f'[{step_name}] before={len(df_before):>7,} | after={len(df_after):>7,} | removed={removed:>6,} ({pct:.1f}%)')

## Inspect random titles BEFORE cleaning

In [ ]:
for grp in ['layoffs', 'hiring', 'wages', 'uncertainty']:
    show_random_titles(news_raw, query_group=grp, n=10, label='(BEFORE cleaning)')

## Cleaning — Step by Step

In [ ]:
# Step 1: Filter English
keep_lang = rules['filters']['language']['keep']
df1 = news_raw[news_raw['language'] == keep_lang].copy()
print_step(news_raw, df1, 'Step 1: filter language=English')

In [ ]:
# Step 2: Drop missing title
df2 = df1.dropna(subset=['title'])
print_step(df1, df2, 'Step 2: drop missing title')

In [ ]:
# Step 3: Drop missing URL
df3 = df2.dropna(subset=['url'])
print_step(df2, df3, 'Step 3: drop missing URL')

In [ ]:
# Step 4: Deduplicate URL + title + date
dedup_cols = [c for c in ['url', 'title', 'seendate'] if c in df3.columns]
df4 = df3.drop_duplicates(subset=dedup_cols, keep='first').reset_index(drop=True)
print_step(df3, df4, 'Step 4: deduplicate url+title+date')

In [ ]:
# Step 5: Remove too-short titles
min_len = rules['filters']['title']['min_length']
df5 = df4[df4['title'].str.len() >= min_len].copy()
print_step(df4, df5, f'Step 5: min title length={min_len}')

In [ ]:
# Step 6: Exclude known social media domains
exclude_domains = rules.get('exclusions', {}).get('domains', [])
if 'domain' in df5.columns and exclude_domains:
    df6 = df5[~df5['domain'].isin(exclude_domains)].copy()
else:
    df6 = df5.copy()
print_step(df5, df6, 'Step 6: exclude social media domains')

In [ ]:
# Step 7: Create text_for_nlp
text_fields = rules['text_for_nlp']['fields_to_combine']
available_fields = [f for f in text_fields if f in df6.columns]
df6['text_for_nlp'] = df6[available_fields].fillna('').agg(' '.join, axis=1).str.strip()

# Add article_month
if 'seendate' in df6.columns:
    df6['seendate'] = pd.to_datetime(df6['seendate'], errors='coerce')
    df6['article_month'] = df6['seendate'].dt.to_period('M').astype(str)

print(f'text_for_nlp created. Sample:')
print(df6['text_for_nlp'].head(3).to_string())

print(f'\nFinal cleaned shape: {df6.shape}')

In [ ]:
print('=== Sample titles AFTER cleaning ===')
for grp in ['layoffs', 'hiring', 'wages', 'uncertainty']:
    show_random_titles(df6, query_group=grp, n=10, label='(AFTER cleaning)')

In [ ]:
cleaning_report = pd.DataFrame([
    {'step': 'raw', 'rows': len(news_raw)},
    {'step': '1_lang_filter', 'rows': len(df1)},
    {'step': '2_drop_missing_title', 'rows': len(df2)},
    {'step': '3_drop_missing_url', 'rows': len(df3)},
    {'step': '4_dedup', 'rows': len(df4)},
    {'step': '5_min_title_len', 'rows': len(df5)},
    {'step': '6_excl_domains', 'rows': len(df6)},
])
cleaning_report['removed_from_previous'] = cleaning_report['rows'].shift(1) - cleaning_report['rows']

print(cleaning_report.to_string(index=False))

report_path = DRIVE_ROOT / 'outputs' / 'data_quality' / 'news_cleaning_report.csv'
cleaning_report.to_csv(report_path, index=False)
print(f'\nCleaning report saved: {report_path}')

## Save candidate file (before approval)

In [ ]:
candidate_path = DRIVE_ROOT / 'data' / 'interim' / 'news' / 'clean_news_candidate.parquet'
df6.to_parquet(candidate_path, index=False)
print(f'Candidate saved: {candidate_path}')

## Manual Approval Gate

**Before approving, verify:**
1. Sample titles after cleaning are relevant to US labor markets
2. No cleaning step removed an unexpectedly large fraction
3. `text_for_nlp` contains meaningful text
4. `article_month` is populated
5. All 6 query groups have sufficient articles

If rules need adjusting, edit `configs/cleaning_rules_news.yaml` and rerun from Step 1.

In [ ]:
APPROVE_THIS_STEP = False

if not APPROVE_THIS_STEP:
    raise RuntimeError(
        'STOP: Inspect the diagnostics above. '
        'If everything is correct, change APPROVE_THIS_STEP=True and rerun this cell.'
    )

In [ ]:
approved_path = DRIVE_ROOT / 'data' / 'interim' / 'news' / 'clean_news_approved.parquet'
df6.to_parquet(approved_path, index=False)
print(f'Approved news saved: {approved_path}')

approval = {
    'approved': True,
    'notebook': '04_audit_and_clean_news.ipynb',
    'approved_at': datetime.datetime.utcnow().isoformat(),
    'input_artifacts': [str(RAW_NEWS_PATH)],
    'output_artifacts': [str(approved_path), str(report_path)],
    'row_counts': {'news_raw': int(len(news_raw)), 'clean_news_approved': int(len(df6))},
    'warnings': [],
    'notes': ''
}
ap_path = DRIVE_ROOT / 'approvals' / '04_news_cleaning_approved.json'
with open(ap_path, 'w') as f:
    json.dump(approval, f, indent=2)
print(f'Approval saved: {ap_path}')
print('Notebook 04 complete. Proceed to 05_build_news_features.ipynb')